# Agent Building — End to End
## From zero to trained RL agent on GridOps

This notebook is your Meta PyTorch hackathon training manual. By the end you will have built, compared, and trained four kinds of agents on your own GridOps environment.

**What you will learn (in this order):**

1. The agent loop — the one idea that everything else decorates.
2. **Random agent** — the baseline that proves plumbing works.
3. **Heuristic agent** — hand-coded domain knowledge.
4. **LLM agent** — prompt-based reasoning (your Round 1 baseline).
5. **RL fundamentals** — policy, value, advantage, PPO.
6. **PPO from scratch** — ~80 lines, trained on GridOps.
7. **Evaluation harness** — comparing all four agents side-by-side.
8. **Hackathon playbook** — what to build first in 48 hours.

**Mental model:**

```
Random  →  Heuristic  →  LLM  →  RL  →  Hybrid
 dumb     domain        language   learning  everything
          priors        priors     from      together
                                    reward
```

Each step adds one layer of intelligence. If you understand the transitions, you understand agents.

---
## Part 1 — The Agent Loop (the only diagram you need)

```
  ┌──────────────┐   observation    ┌──────────────┐
  │              │ ───────────────► │              │
  │ ENVIRONMENT  │                  │    AGENT     │
  │   (world)    │ ◄─────────────── │   (brain)    │
  │              │      action      │              │
  └──────────────┘                  └──────────────┘
        │                                  ▲
        │             reward               │
        └──────────────────────────────────┘
```

That is it. Every agent — random, heuristic, LLM, RL — fits this loop. The only thing that changes is **how the agent maps observation → action**.

| Agent type | Mapping strategy | Needs training? |
|---|---|---|
| Random | `action = sample()` | No |
| Heuristic | `action = if/else rules` | No |
| LLM | `action = llm(prompt(obs))` | No (pre-trained) |
| RL | `action = neural_net(obs)` | **Yes — thousands of episodes** |

The **reward** is the only feedback signal. An RL agent uses it to improve. The others ignore it (they just run and get scored).

---
## Part 2 — Setup: Your GridOps environment

We will use the actual GridOps env you built. No servers, no docker — just import it in-process.

In [ ]:
import sys, os, json
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))

from gridops.server.environment import GridOpsEnvironment
from gridops.models import GridOpsAction, GridOpsObservation
from gridops.tasks.graders import grade_episode

env = GridOpsEnvironment()
obs = env.reset(seed=84, task_id='task_1_normal')

print('Observation keys:', list(obs.model_dump().keys())[:8], '...')
print(f'Hour: {obs.hour}, Demand: {obs.demand_kw:.0f} kW, Solar: {obs.solar_kw:.0f} kW')
print(f'Battery SOC: {obs.battery_soc*100:.0f}%, Grid price: Rs {obs.grid_price:.1f}/kWh')

Observation keys: ['done', 'reward', 'metadata', 'hour', 'demand_kw', 'solar_kw', 'battery_soc', 'grid_price', 'diesel_fuel_remaining', 'diesel_is_on', 'demand_forecast_4h', 'solar_forecast_4h', 'price_forecast_4h', 'cumulative_blackout_kwh', 'cumulative_cost', 'day_of_episode', 'blackout_this_step', 'cost_this_step', 'grid_kw_this_step', 'narration', 'flow_solar', 'flow_grid_import', 'flow_grid_export', 'flow_battery_discharge', 'flow_battery_charge', 'flow_diesel', 'flow_demand', 'flow_blackout', 'flow_shed', 'flow_total_supply', 'flow_total_consumption']
Hour: 0.0, Demand: 65 kW, Solar: 0 kW
Battery SOC: 50%, Grid price: Rs 7.8/kWh


### The action space

GridOps has **3 continuous actions**, each bounded to a specific range:

| Action | Range | What it does |
|---|---|---|
| `battery_dispatch` | -1.0 to +1.0 | Charge (−) or discharge (+) battery, 100 kW max |
| `diesel_dispatch` | 0.0 to 1.0 | Diesel generator output, 0–100 kW |
| `demand_shedding` | 0.0 to 1.0 | Ask residents to reduce (0–20%) |

The grid is a **slack variable** — it automatically absorbs the residual. You do not control it directly.

This is crucial to understand: your agent has 3 knobs, and the grid does the rest (within its ±200 kW limit).

---
## Part 3 — Agent #1: Random

The dumbest possible agent. **Why build it?** Because if your random agent crashes, your training loop is broken. Random agents verify plumbing.

A random agent is also your absolute floor — anything trained should beat it.

In [ ]:
def random_agent(obs):
    return GridOpsAction(
        battery_dispatch=np.random.uniform(-1, 1),
        diesel_dispatch=np.random.uniform(0, 1),
        demand_shedding=np.random.uniform(0, 1),
    )

def run_episode(env, policy_fn, task_id='task_1_normal', seed=42, verbose=False):
    """Run one full episode and return trajectory + grade.

    NOTE: GridOps env.step() returns the Observation DIRECTLY (not a wrapped
    result). The Observation carries `reward`, `done`, `metadata` via the
    openenv base class, alongside the domain-specific fields.
    """
    obs = env.reset(seed=seed, task_id=task_id)
    trajectory = []
    for step in range(72):
        action = policy_fn(obs)
        obs = env.step(action)  # observation IS the result
        trajectory.append({
            'hour': obs.hour, 'demand': obs.demand_kw, 'solar': obs.solar_kw,
            'soc': obs.battery_soc, 'price': obs.grid_price,
            'blackout': obs.flow_blackout, 'cost': obs.cost_this_step,
            'reward': obs.reward or 0.0,
        })
        if obs.done:
            break
    grade = env.state.grade  # env.state is a property, not a method
    return trajectory, grade

traj, grade = run_episode(env, random_agent)
total_reward = sum(t['reward'] for t in traj)
total_blackout = sum(t['blackout'] for t in traj)
print(f'Random agent — total reward: {total_reward:.2f}')
print(f'Total blackout: {total_blackout:.1f} kWh  (this is bad!)')
print(f'Grade: {grade}')

**Observation:** the random agent will likely cause blackouts (running diesel randomly, shedding demand randomly) and score near zero on the grader. That is expected. The grader has a **reliability gate** — below 90% reliability, score collapses to 0.

This tells us: **blackouts are catastrophic.** Our policy must prioritize keeping the lights on.

---
## Part 4 — Agent #2: Heuristic (domain knowledge)

Now encode what a human microgrid operator actually knows:

1. **Night (0–6h)**: cheap grid, low demand → charge battery
2. **Day (6–15h)**: solar surplus → let it spill to grid (export) or charge battery
3. **Pre-peak (15–17h)**: top up battery for the evening squeeze
4. **Evening peak (18–22h)**: discharge battery to cover the demand gap
5. **Diesel**: only as last resort when battery is empty and demand > grid cap

This is a **rule-based policy**. Fast. Interpretable. Decent on Task 1. Breaks on Task 3 because it cannot adapt.

In [ ]:
def heuristic_agent(obs):
    """Rule-based policy for GridOps.

    CRITICAL: GridOps episodes start at 6 AM. obs.hour is the STEP INDEX
    (0..71), not a wall-clock hour. Conversion:
        step_in_day  = int(obs.hour) % 24
        wall_clock_h = (step_in_day + 6) % 24
    So step 0 = 6 AM, step 14 = 8 PM (evening peak), step 18 = midnight.
    """
    step_in_day = int(obs.hour) % 24
    soc = obs.battery_soc
    demand = obs.demand_kw
    solar = obs.solar_kw
    price = obs.grid_price

    battery = 0.0
    diesel = 0.0
    shed = 0.0

    # Windows in STEP INDEX terms (not wall clock):
    #   18-23 = night (12 AM – 5 AM): cheap grid, demand ~50 kW → charge hard
    #   0-5   = morning (6 AM – 11 AM): demand ramps up, solar ramping up
    #   6-11  = midday (12 PM – 5 PM): solar peaks, cheap-ish energy → top off
    #   12-17 = EVENING PEAK (6 PM – 11 PM): 200-250 kW → discharge battery

    if 18 <= step_in_day <= 23:
        # Night: charge with cheap grid
        if soc < 0.95:
            battery = -0.9

    elif 6 <= step_in_day <= 10:
        # Midday: if solar > demand, passively let battery charge from surplus
        # (grid will export the rest). If solar < demand but battery < 80%,
        # gently charge from grid while prices are still moderate.
        if soc < 0.80 and solar < demand:
            battery = -0.3

    elif step_in_day == 11:
        # Pre-peak top-up — the peak is coming in 1 hour
        if soc < 0.95:
            battery = -0.5

    elif 12 <= step_in_day <= 17:
        # Evening peak — discharge battery to cover the gap over grid cap
        # Grid cap = 200 kW, so anything above that must come from battery/diesel
        gap = demand - solar - 200.0
        if gap > 0:
            battery = float(np.clip(gap / 100.0, 0.0, 1.0))
        # Diesel only as last resort: battery depleted AND big gap AND price awful
        if soc < 0.15 and gap > 50 and price > 12:
            diesel = 0.8
        # Emergency shedding if we're about to black out
        if soc < 0.10 and gap > 30:
            shed = 0.3

    return GridOpsAction(
        battery_dispatch=float(np.clip(battery, -1, 1)),
        diesel_dispatch=float(np.clip(diesel, 0, 1)),
        demand_shedding=float(np.clip(shed, 0, 1)),
    )

traj_h, grade_h = run_episode(env, heuristic_agent)
print(f'Heuristic — reward: {sum(t["reward"] for t in traj_h):.2f}')
print(f'Blackout: {sum(t["blackout"] for t in traj_h):.1f} kWh')
print(f'Grade: {grade_h}')

### Visualize what the heuristic is doing

A plot is worth a thousand reward numbers. This shows SOC, demand, and solar over the full 72-hour episode.

In [ ]:
def plot_trajectory(traj, title='Trajectory'):
    hours = [t['hour'] for t in traj]
    fig, ax1 = plt.subplots(figsize=(12, 4))
    ax1.plot(hours, [t['demand'] for t in traj], 'r-', label='Demand (kW)', alpha=0.7)
    ax1.plot(hours, [t['solar'] for t in traj], 'gold', label='Solar (kW)', alpha=0.7)
    ax1.fill_between(hours, 0, [t['blackout'] for t in traj], color='red', alpha=0.5, label='Blackout (kW)')
    ax1.set_xlabel('Hour')
    ax1.set_ylabel('kW')
    ax1.legend(loc='upper left')
    ax2 = ax1.twinx()
    ax2.plot(hours, [t['soc']*100 for t in traj], 'b-', linewidth=2, label='Battery SOC (%)')
    ax2.set_ylabel('SOC (%)')
    ax2.set_ylim(0, 100)
    ax2.legend(loc='upper right')
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_trajectory(traj_h, 'Heuristic agent on Task 1')

**What to look for:**
- Does SOC rise at night and fall in the evening? (should be yes)
- Does blackout stay at zero? (should be yes on Task 1)
- Does the agent pre-charge before Day 2's heatwave? (run on Task 2 to check — it probably will not, which is why heuristics fail there)

---
## Part 5 — Agent #3: LLM (your Round 1 inference.py, explained)

An LLM agent turns the observation into text, asks the model to reason, and parses an action back out. This is what your `inference.py` already does.

**Why it works:** Large language models have read millions of words about energy, markets, planning. They have priors you did not hand-code.

**Why it is limited:** Slow (seconds per step), expensive (API costs), non-deterministic (same obs → different action), cannot directly learn from reward.

### The pattern

```
obs (pydantic)  →  format as text  →  LLM call  →  parse JSON  →  GridOpsAction
```

Exactly what your `inference.py` does. Here is the core loop, distilled:

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=os.getenv('API_BASE_URL', 'https://router.huggingface.co/v1'),
    api_key=os.getenv('HF_TOKEN', os.getenv('API_KEY', 'dummy')),
)
MODEL = os.getenv('MODEL_NAME', 'meta-llama/Llama-3.3-70B-Instruct')

SYSTEM = (
    "You are an expert microgrid operator. Output JSON with 3 fields:\n"
    "{\"battery_dispatch\": -1 to 1, \"diesel_dispatch\": 0 to 1, \"demand_shedding\": 0 to 1}\n"
    "Strategy: charge battery at night (cheap grid), discharge at evening peak.\n"
    "Diesel only when battery < 15% AND price > 15. Avoid blackouts at all cost."
)

def llm_agent(obs):
    prompt = (
        f"Hour {obs.hour:.0f}, Day {obs.day_of_episode}\n"
        f"Demand: {obs.demand_kw:.0f} kW, Solar: {obs.solar_kw:.0f} kW\n"
        f"Battery SOC: {obs.battery_soc*100:.0f}%, Price: Rs {obs.grid_price:.1f}\n"
        f"Price forecast 4h: {[f'{p:.1f}' for p in obs.price_forecast_4h]}\n"
        f"What action? JSON only."
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'system', 'content': SYSTEM},
                  {'role': 'user', 'content': prompt}],
        temperature=0.1,
        max_tokens=200,
    )
    text = resp.choices[0].message.content
    try:
        start, end = text.find('{'), text.rfind('}') + 1
        data = json.loads(text[start:end])
    except Exception:
        data = {'battery_dispatch': 0, 'diesel_dispatch': 0, 'demand_shedding': 0}
    return GridOpsAction(
        battery_dispatch=float(np.clip(data.get('battery_dispatch', 0), -1, 1)),
        diesel_dispatch=float(np.clip(data.get('diesel_dispatch', 0), 0, 1)),
        demand_shedding=float(np.clip(data.get('demand_shedding', 0), 0, 1)),
    )

# Optional: run if you have API key set. 72 LLM calls can be slow/costly.
# traj_llm, grade_llm = run_episode(env, llm_agent)
# print(f'LLM grade: {grade_llm}')

**Observation — the Pareto frontier:**

| Agent | Speed | Cost | Task 1 | Task 3 | Improves? |
|---|---|---|---|---|---|
| Random | 0.1 ms | $0 | ~0 | 0 | No |
| Heuristic | 0.1 ms | $0 | 0.7 | ~0.3 | No |
| LLM | ~2s/step | $$$ | 0.6–0.8 | 0.4 | No |
| **RL (PPO)** | **0.5 ms** | **$0 at inference** | **0.9+** | **0.5–0.7** | **Yes** |

RL dominates the frontier **once trained**. Training is the price you pay.

---
## Part 6 — RL Fundamentals (the minimum you need to know)

Before we code PPO, three concepts. No more.

### Concept 1 — The Policy π(a | s)

A function that takes a state `s` and outputs either (a) a specific action (deterministic) or (b) a probability distribution over actions (stochastic).

For **continuous** actions like ours (battery_dispatch is a real number), the stochastic policy outputs `mean` and `std`, and we sample from `Normal(mean, std)`.

```
obs  →  neural_net  →  (mean, std)  →  sample Normal  →  action
```

### Concept 2 — The Value function V(s)

“How good is it to be in this state, assuming I follow my current policy from here on?”

V(s) is a scalar. Another neural net predicts it. We train it by regression toward actual observed returns.

### Concept 3 — The Advantage A(s, a)

“Was action `a` better than average in state `s`?”

```
A(s, a)  =  actual_return_after_taking_a  −  V(s)
```

- A > 0 → action was better than expected → make it more likely
- A < 0 → action was worse than expected → make it less likely

**That is the entire learning signal.** Every RL algorithm is some variant of "nudge the policy toward high-advantage actions."

### Why PPO?

Vanilla policy gradient has a bug: if you take a big gradient step, the new policy can be drastically different from the data-collection policy, and the gradient estimate becomes wrong.

PPO fixes this with a **clip**: updates are capped so the new policy stays close to the old one. Three lines of code. Meta trains language models (RLHF) with this. It just works.

---
## Part 7 — Agent #4: PPO from scratch on GridOps

**Proximal Policy Optimization (PPO)** is the algorithm that sits behind most practical RL deployments today — from OpenAI's game-playing agents to Meta's RLHF-trained language models. "From scratch" here means we implement every piece ourselves in ~100 lines of PyTorch, with no black-box RL library in the way. This lets you see exactly what is happening and modify it confidently.

The build has four concrete steps that must happen in order, because each step provides something the next one depends on:

| Step | What we build | Why it is needed |
|---|---|---|
| 7.1 | **Gym wrapper** | Neural nets require fixed-length float arrays. GridOps gives us a rich pydantic object. We translate between the two. |
| 7.2 | **Actor-Critic network** | The neural net that represents our policy. It has two outputs: *what action to take* and *how good this state is*. |
| 7.3 | **Rollout collection + GAE** | We run the current policy to gather experience, then compute how good each action was relative to expectation. |
| 7.4 | **PPO update** | We use the collected experience to improve the policy — carefully, so one bad batch of data does not destroy prior learning. |

The full training loop is then simply: **collect → compute advantages → update → repeat**.

---
### Step 7.1 — Observation wrapper: pydantic object → float vector

A neural network is fundamentally a function from `ℝⁿ → ℝᵐ` — it only knows how to process fixed-size arrays of floats. GridOps observations are structured pydantic objects with named fields, forecasts of different lengths, and physical units spanning wildly different scales (hours, kilowatts, rupees, fractions).

The wrapper does two things:

**Flattening** — extract every relevant field into a single 1-D array with a fixed length (`OBS_DIM = 20`). Forecasts that might be shorter than 4 steps are zero-padded so the shape never changes.

**Normalization** — divide each value by a typical maximum (e.g. `demand_kw / 500.0`). This maps all inputs to roughly the range `[0, 1]`. Without this, a gradient step that learns something about `battery_soc` (always 0–1) would be wildly miscalibrated when it encounters `demand_kw` values in the hundreds. Normalization is not optional — training will fail to converge without it.

On the action side, the network outputs values in `[-1, 1]` (because we use `tanh` activation). The wrapper re-maps them to the valid ranges each GridOps action expects.

In [ ]:
BATTERY_CAPACITY_KWH = 500.0   # matches gridops.simulation.physics

class GymGridOps:
    """Thin wrapper converting GridOps → fixed-size float vectors.

    V4 design (post-Gemini review):
      - Cyclical time encoding: sin/cos(2π · clock_hour / 24) replaces linear
        `hour/72`. The MLP can now read "it's 3 AM" directly.
      - 24-hour forecasts from the env (demand/solar/price), with noise that
        grows with horizon. The evening peak 16 hours ahead is now visible
        at night — arbitrage is learnable without recurrence.
      - History buffer dropped: the 24h forecast subsumes short-term trend info.
      - randomize_initial_soc: unchanged from V3. Training-only perturbation
        to avoid the "always-drain" local optimum.

    The wrapper reads forecasts from `obs.*_forecast_24h`, so the same
    flattening works for training (uses internal env) AND evaluation
    (uses a separately-constructed env via run_episode).
    """
    FORECAST_H = 24
    # Scalars: hour/72, sin_t, cos_t, demand, solar, soc, price, fuel, diesel_on, day
    BASE_DIM = 10
    OBS_DIM = BASE_DIM + 3 * FORECAST_H      # 10 + 72 = 82
    ACT_DIM = 3

    def __init__(self, task_id='task_1_normal', randomize_initial_soc=False):
        self.env = GridOpsEnvironment()
        self.task_id = task_id
        self.randomize_initial_soc = randomize_initial_soc

    @staticmethod
    def _base_features(obs):
        clock = (int(obs.hour) + 6) % 24         # episode starts at 6 AM
        theta = 2.0 * np.pi * clock / 24.0
        scalars = [
            obs.hour / 72.0,
            float(np.sin(theta)),
            float(np.cos(theta)),
            obs.demand_kw / 500.0,
            obs.solar_kw / 250.0,
            obs.battery_soc,
            obs.grid_price / 20.0,
            obs.diesel_fuel_remaining,
            float(obs.diesel_is_on),
            obs.day_of_episode / 3.0,
        ]
        def _pad(lst, n):
            vals = list(lst or [])
            if len(vals) >= n:
                return vals[:n]
            return vals + [vals[-1] if vals else 0.0] * (n - len(vals))

        fc = []
        for lst, scale in [
            (obs.demand_forecast_24h, 500.0),
            (obs.solar_forecast_24h, 250.0),
            (obs.price_forecast_24h, 20.0),
        ]:
            fc.extend([x / scale for x in _pad(lst, GymGridOps.FORECAST_H)])
        return scalars + fc

    @classmethod
    def flatten(cls, obs):
        """Observation → float32 vector. Stateless — use for eval too."""
        return np.array(cls._base_features(obs), dtype=np.float32)

    def reset(self, seed=42):
        obs = self.env.reset(seed=seed, task_id=self.task_id)
        if self.randomize_initial_soc:
            initial_soc = float(np.random.uniform(0.10, 0.95))
            self.env._micro.battery_soc_kwh = initial_soc * BATTERY_CAPACITY_KWH
            obs.battery_soc = initial_soc
        return self.flatten(obs)

    def step(self, action_vec):
        a = GridOpsAction(
            battery_dispatch=float(np.clip(action_vec[0], -1, 1)),
            diesel_dispatch=float(np.clip((action_vec[1] + 1) / 2, 0, 1)),
            demand_shedding=float(np.clip((action_vec[2] + 1) / 2, 0, 1)),
        )
        obs = self.env.step(a)
        reward = float(obs.reward or 0.0)
        # No wrapper penalty: env reward already reflects step_cost (incl. VoLL).
        done = bool(obs.done)
        return self.flatten(obs), reward, done, {}

g = GymGridOps()
v = g.reset()
print(f'OBS_DIM = {GymGridOps.OBS_DIM}, obs shape: {v.shape}')
print(f'First 10 (scalars): {v[:10].round(3)}')
print(f'Demand 24h forecast (normalized): {v[10:34].round(2)}')

# Verify forecasts read from obs match across fresh envs (eval-time safety)
obs = g.env.reset(seed=7, task_id='task_1_normal')
flat = GymGridOps.flatten(obs)
print(f'Stateless flatten works on a fresh obs. shape={flat.shape}, first 5={flat[:5].round(3)}')


### Step 7.2 — Actor-Critic network

The **Actor-Critic** architecture is the standard design for continuous-action RL. It combines two roles in one network:

- The **Actor** (policy head) — decides *what to do*. Given the current state, it outputs a probability distribution over actions. For continuous actions, this is a **Gaussian (Normal) distribution** parameterised by a mean `μ` and standard deviation `σ`. The agent samples an action from this distribution, which provides the exploration we need during training.

- The **Critic** (value head) — estimates *how good the current situation is*. It outputs a single scalar `V(s)`, the expected cumulative reward from state `s` onward under the current policy. The critic does not directly take actions — it exists purely to guide the actor's learning signal.

**Why share a trunk?** The two heads share a common MLP backbone (the "trunk"). This is motivated by the intuition that understanding the state well enough to predict value and understanding it well enough to choose actions require similar representations. Shared weights mean both heads benefit from each other's gradients and the network is smaller.

**Why Gaussian actions?** Our action space is continuous (battery dispatch is a real number in [-1, 1], not a discrete choice). A Gaussian lets us express both *intent* (the mean is the agent's best guess) and *uncertainty* (a large std means "I am not sure yet, explore more"). As training progresses, the std typically shrinks — the policy becomes more decisive.

**`log_std` as a learnable parameter, not a head:** The standard deviation is represented as a single learnable vector (not state-dependent). This is a deliberate simplification — it keeps exploration consistent across states and avoids instability from the network over-shrinking std early in training.

```
obs (20 floats)
     │
  [Linear → Tanh → Linear → Tanh]   ← shared trunk (feature extraction)
     │
     ├──► [Linear → tanh]  → mean μ       (actor head)
     │    log_std (parameter) → std σ
     │    action ~ Normal(μ, σ)
     │
     └──► [Linear]          → V(s)        (critic head, scalar)
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

class ActorCritic(nn.Module):
    def __init__(self, obs_dim=GymGridOps.OBS_DIM, act_dim=3, hidden=256):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.mean_head = nn.Linear(hidden, act_dim)
        self.value_head = nn.Linear(hidden, 1)
        # Log-std is a learnable parameter, not a state-dependent head (standard PPO trick)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)

    def forward(self, x):
        h = self.trunk(x)
        mean = torch.tanh(self.mean_head(h))  # bound to [-1, 1]
        value = self.value_head(h).squeeze(-1)
        return mean, self.log_std.exp().expand_as(mean), value

    def act(self, obs):
        """Sample action + return log_prob + value. Called at rollout time."""
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(-1)
        return action, log_prob, value

    def evaluate(self, obs, actions):
        """Used during PPO update to re-evaluate stored actions with updated policy."""
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(actions).sum(-1)
        entropy = dist.entropy().sum(-1)
        return log_prob, entropy, value

net = ActorCritic()
print(f'Network params: {sum(p.numel() for p in net.parameters()):,}')
print(f'obs_dim in use: {net.trunk[0].in_features}')

### Step 7.3 — Rollout collection and Generalized Advantage Estimation (GAE)

#### What is a rollout?

A **rollout** is a chunk of experience: the agent plays through `N` environment steps using its *current* policy and records everything it did and saw. Concretely, for each step `t` we store:

- `obs_t` — the state the agent was in
- `action_t` — what it sampled from the policy
- `reward_t` — what the environment gave back
- `log_prob_t` — the log-probability of that action under the *current* policy (needed for PPO's ratio later)
- `value_t` — the critic's estimate `V(s_t)` (needed for advantage computation)

After `N` steps, we have a dataset we can learn from. We then need to answer: **was each action good or bad?**

#### Computing the advantage: was this action better than expected?

The **advantage** `A_t` answers: "relative to what I expected from this state, how much better (or worse) did taking this action turn out to be?"

A naïve approach is to compare the actual return (total discounted reward from step `t` to end) with `V(s_t)`. But this has high **variance** — a single lucky or unlucky episode can swing the estimate wildly. We want a more stable signal.

**Generalized Advantage Estimation (GAE)** solves this by computing a smoothly-blended estimate using a chain of **TD errors**. The TD error at step `t` is:

```
δ_t = r_t + γ · V(s_{t+1}) − V(s_t)
```

This is the single-step surprise: "I expected `V(s_t)` from here, but I actually got `r_t` and then landed in a state worth `V(s_{t+1})`." Positive means the step was better than expected.

GAE then combines these TD errors with exponentially-decaying weights:

```
A_t = δ_t + (γλ)·δ_{t+1} + (γλ)²·δ_{t+2} + ...
```

The two hyperparameters control the bias-variance tradeoff:

| Parameter | Typical value | What it controls |
|---|---|---|
| `γ` (gamma) | 0.99 | **Discount factor** — how much to value future rewards. At 0.99, a reward 100 steps away is worth `0.99¹⁰⁰ ≈ 0.37` of an immediate reward. |
| `λ` (lambda) | 0.95 | **GAE smoothing** — at `λ=0`, only the immediate TD error matters (low variance, high bias). At `λ=1`, it sums all future TD errors (low bias, high variance, equivalent to Monte Carlo). `λ=0.95` is the standard empirical sweet spot. |

The **return** for the value function loss is simply `A_t + V(s_t)` — the advantage plus the baseline. This is the target the critic is trained to predict.

In [ ]:
def collect_rollout(net, env_sampler, n_steps=288, gamma=0.99, lam=0.95):
    """Collect n_steps of experience across multi-task episodes.

    env_sampler() -> (GymGridOps, task_id_str). Called at episode boundaries.
    We also track which task each step came from so the PPO update can
    normalize advantages per-task (prevents high-variance crisis episodes
    from dominating the gradient).
    """
    obs_buf, act_buf, rew_buf, done_buf = [], [], [], []
    val_buf, logp_buf, task_buf = [], [], []

    env, task_id = env_sampler()
    obs = env.reset(seed=int(np.random.randint(1_000_000)))
    for _ in range(n_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            action, log_prob, value = net.act(obs_t)
        action_np = action.squeeze(0).numpy()
        next_obs, reward, done, _ = env.step(action_np)

        obs_buf.append(obs)
        act_buf.append(action_np)
        rew_buf.append(reward)
        done_buf.append(float(done))
        val_buf.append(value.item())
        logp_buf.append(log_prob.item())
        task_buf.append(task_id)

        if done:
            env, task_id = env_sampler()
            obs = env.reset(seed=int(np.random.randint(1_000_000)))
        else:
            obs = next_obs

    # Final bootstrap value (for the last state)
    with torch.no_grad():
        _, _, last_val = net.forward(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0))
    last_val = last_val.item()

    # GAE
    advantages = np.zeros(n_steps, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(n_steps)):
        next_val = last_val if t == n_steps - 1 else val_buf[t+1]
        next_nonterminal = 1.0 - done_buf[t]
        delta = rew_buf[t] + gamma * next_val * next_nonterminal - val_buf[t]
        gae = delta + gamma * lam * next_nonterminal * gae
        advantages[t] = gae
    returns = advantages + np.array(val_buf, dtype=np.float32)

    return {
        'obs':     torch.as_tensor(np.array(obs_buf), dtype=torch.float32),
        'actions': torch.as_tensor(np.array(act_buf), dtype=torch.float32),
        'log_probs': torch.as_tensor(logp_buf, dtype=torch.float32),
        'advantages': torch.as_tensor(advantages, dtype=torch.float32),
        'returns': torch.as_tensor(returns, dtype=torch.float32),
        'task_ids': task_buf,
        'total_reward': float(sum(rew_buf)),
    }

### Step 7.4 — The PPO update

We have a rollout with advantages. Now we improve the policy. The core challenge: gradient descent on policy data can take too large a step, landing us in a region where the policy is completely different — and then the advantages computed from the *old* policy are no longer valid estimates for the *new* one. Naive policy gradient breaks here.

**PPO's solution: the probability ratio + clip.**

For each stored action, compute how much more (or less) likely the *updated* policy would be to take that action compared to the *old* policy that collected the data:

```
ratio_t = π_new(a_t | s_t) / π_old(a_t | s_t)
        = exp(log π_new − log π_old)   ← numerically stable form
```

A ratio of `1.0` means the policies agree. `> 1` means the new policy prefers this action more. `< 1` means less.

The **clipped surrogate objective** caps how much this ratio can push the gradient:

```
surr1 = ratio_t × A_t                               ← unclipped
surr2 = clip(ratio_t, 1−ε, 1+ε) × A_t              ← clipped

policy_loss = −mean( min(surr1, surr2) )
```

If advantage is positive (action was good), `surr1` wants to increase the ratio (make this action more likely). But `surr2` caps the ratio at `1+ε`, so no single batch of data can push the policy too far. If advantage is negative (action was bad), the clip prevents the policy from collapsing too fast in the other direction. In either case, the `min` ensures we take the pessimistic (conservative) estimate.

**The full loss has three terms:**

| Term | Coefficient | Purpose |
|---|---|---|
| `policy_loss` | 1.0 | Improve the actor toward high-advantage actions |
| `value_loss` | 0.5 | Train the critic to better predict `V(s)` (MSE against observed returns) |
| `−entropy_bonus` | −0.01 | Penalize certainty — a small negative on entropy encourages the policy to keep exploring rather than collapsing to a single deterministic action too early |

**Advantage normalization** (subtracting the mean and dividing by std before the update) is a practical stability trick — it ensures the magnitude of gradient updates does not depend on the arbitrary scale of your reward function.

**Gradient clipping** (`clip_grad_norm_ = 0.5`) prevents any single parameter update from being catastrophically large, which can otherwise send the network to a degenerate region it never recovers from.

The update runs for `K=4` epochs over the same rollout data, shuffled into minibatches. More passes squeeze more signal from each rollout without collecting new data, but too many epochs violate the on-policy assumption (the clip is there precisely to manage this tradeoff).

In [ ]:
def ppo_update(net, optimizer, rollout, clip_eps=0.2, epochs=4, batch_size=64):
    obs, actions = rollout['obs'], rollout['actions']
    old_log_probs = rollout['log_probs']
    advantages = rollout['advantages'].clone()
    returns = rollout['returns']
    task_ids = rollout['task_ids']

    # Per-task advantage normalization
    # -------------------------------------------------------------
    # Crisis episodes produce much larger penalties (big blackouts → big negative rewards)
    # than normal episodes. If we normalize advantages globally, crisis dominates the
    # gradient and the policy forgets how to handle normal/heatwave conditions.
    # Normalizing within each task equalizes the signal so all tasks contribute equally.
    task_arr = np.array(task_ids)
    for task in set(task_ids):
        mask_np = (task_arr == task)
        if mask_np.sum() > 1:
            idx = torch.as_tensor(np.where(mask_np)[0], dtype=torch.long)
            task_adv = advantages[idx]
            advantages[idx] = (task_adv - task_adv.mean()) / (task_adv.std() + 1e-8)

    n = obs.shape[0]
    for _ in range(epochs):
        idx = torch.randperm(n)
        for start in range(0, n, batch_size):
            mb = idx[start:start+batch_size]
            log_probs, entropy, values = net.evaluate(obs[mb], actions[mb])

            ratio = (log_probs - old_log_probs[mb]).exp()
            surr1 = ratio * advantages[mb]
            surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages[mb]
            policy_loss = -torch.min(surr1, surr2).mean()

            value_loss = F.mse_loss(values, returns[mb])
            entropy_bonus = entropy.mean()

            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            optimizer.step()

    return {'policy_loss': policy_loss.item(), 'value_loss': value_loss.item(),
            'entropy': entropy_bonus.item()}

---
## Part 8 — Training (V2: multi-task, LR decay, best-checkpoint)

We now loop: collect rollout (across all 3 tasks) → update → evaluate → keep-if-best → repeat.

**Four upgrades from V1:**

1. **Multi-task rollouts** — each episode samples a task uniformly from `{normal, heatwave, crisis}`. The policy learns to handle all three jointly.
2. **Per-task advantage normalization** (in `ppo_update`) — prevents crisis episodes from dominating the gradient and erasing what the policy learned on easier tasks.
3. **Linear LR decay** — starts at `3e-4`, ends at `1e-5`. Late-stage drift was the cause of the catastrophic forgetting we saw at iter 3030 in V1.
4. **Best-checkpoint eval** — every 50 iterations we evaluate on held-out seeds across all tasks, and keep only the best-scoring weights. At the end we restore them, so the final model is whatever peaked, not whatever came last.

**Expectation:** eval score climbs steadily and plateaus. If it drops for 5+ consecutive eval points, the LR is still too high late in training. If it never climbs, check observation normalization or reward scale.

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

net = ActorCritic()
opt = torch.optim.Adam(net.parameters(), lr=3e-4)

TASKS = ['task_1_normal', 'task_2_heatwave', 'task_3_crisis']
P_START = np.array([1/3, 1/3, 1/3])
P_END   = np.array([0.20, 0.30, 0.50])

def make_env_sampler(probs, randomize_soc=True):
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum()
    def sampler():
        task = TASKS[int(np.random.choice(len(TASKS), p=probs))]
        return GymGridOps(task_id=task, randomize_initial_soc=randomize_soc), task
    return sampler

def quick_eval(net, n_seeds=2):
    """Evaluate current policy on all 3 tasks at FIXED initial SOC (50%).
    Randomization is only for training — eval must be deterministic so
    scores are comparable across runs."""
    net.eval()
    scores = []
    for task in TASKS:
        for seed in range(n_seeds):
            flat_env = GymGridOps(task_id=task, randomize_initial_soc=False)
            obs = flat_env.reset(seed=seed)
            done = False
            while not done:
                with torch.no_grad():
                    mean, _, _ = net.forward(torch.as_tensor(obs).unsqueeze(0))
                obs, _, done, _ = flat_env.step(mean.squeeze(0).numpy())
            grade = flat_env.env.state.grade
            scores.append(grade.get('score', 0.0) if grade else 0.0)
    net.train()
    return float(np.mean(scores))

# -----------------------------------------------------------------
# Hyperparameters
N_ITERATIONS = 3000
ROLLOUT_STEPS = 864
LR_START, LR_END = 3e-4, 1e-5
EVAL_EVERY = 50
# -----------------------------------------------------------------

reward_history, eval_history, probs_history = [], [], []
best_score = -float('inf')
best_state = None
best_iter = 0

for it in range(N_ITERATIONS):
    frac = it / max(1, N_ITERATIONS - 1)

    lr = LR_START + (LR_END - LR_START) * frac
    for g in opt.param_groups:
        g['lr'] = lr

    probs = P_START + (P_END - P_START) * frac
    env_sampler = make_env_sampler(probs, randomize_soc=True)   # V3: randomize initial SOC during training

    rollout = collect_rollout(net, env_sampler, n_steps=ROLLOUT_STEPS)
    stats = ppo_update(net, opt, rollout)
    reward_history.append(rollout['total_reward'] / (ROLLOUT_STEPS / 72))
    probs_history.append(probs.copy())

    if (it + 1) % EVAL_EVERY == 0:
        score = quick_eval(net)
        eval_history.append((it + 1, score))
        is_best = score > best_score
        if is_best:
            best_score = score
            best_iter = it + 1
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        tag = ' ★ BEST' if is_best else ''
        print(f'Iter {it+1:4d} | reward: {reward_history[-1]:7.2f} '
              f'| eval: {score:.3f} | best: {best_score:.3f} @ {best_iter} '
              f'| p: [{probs[0]:.2f} {probs[1]:.2f} {probs[2]:.2f}] '
              f'| lr: {lr:.1e}{tag}')

if best_state is not None:
    net.load_state_dict(best_state)
    print(f'\nRestored best checkpoint: eval score {best_score:.3f} at iter {best_iter}')

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(11, 5.5),
                                     gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax_top.plot(reward_history, color='tab:blue', alpha=0.5, label='train reward (avg/episode)')
ax_top.set_ylabel('Avg episode reward', color='tab:blue')
ax_top.grid(alpha=0.3)
if eval_history:
    its, scs = zip(*eval_history)
    axR = ax_top.twinx()
    axR.plot(its, scs, 'o-', color='tab:red', label='eval score (mean over 3 tasks)')
    axR.set_ylabel('Eval score (0-1)', color='tab:red')
    axR.set_ylim(0, 1)

probs_arr = np.array(probs_history)
ax_bot.plot(probs_arr[:, 0], label='task_1_normal',   color='tab:green')
ax_bot.plot(probs_arr[:, 1], label='task_2_heatwave', color='tab:orange')
ax_bot.plot(probs_arr[:, 2], label='task_3_crisis',   color='tab:red')
ax_bot.set_xlabel('Iteration'); ax_bot.set_ylabel('Task sample prob')
ax_bot.set_ylim(0, 0.6); ax_bot.grid(alpha=0.3); ax_bot.legend(loc='upper left', fontsize=8)
plt.suptitle('PPO — multi-task curriculum + randomized initial SOC (V3)')
plt.tight_layout(); plt.show()

**What a good curve looks like:**

- Starts negative (random behaviour → blackouts, wasted diesel)
- Climbs steadily for the first 20–30 iterations
- Plateaus as the policy converges
- Small wiggles are normal (PPO is noisy)

If it is flat: your env flattening or reward is wrong. If it is oscillating wildly: lower LR or smaller clip.

---
## Part 9 — Evaluation harness: the four agents head-to-head

This is the plot that wins the hackathon demo. It shows that:
1. You understand the progression.
2. Your RL agent beats hand-coded rules.
3. You can generalize across tasks.

In [ ]:
class PPOAgent:
    """Stateless adapter: GridOpsObservation → GridOpsAction via the net."""
    def __init__(self, net):
        self.net = net

    def __call__(self, obs):
        flat = GymGridOps.flatten(obs)
        with torch.no_grad():
            mean, _, _ = self.net.forward(torch.as_tensor(flat).unsqueeze(0))
        action_vec = mean.squeeze(0).numpy()
        return GridOpsAction(
            battery_dispatch=float(np.clip(action_vec[0], -1, 1)),
            diesel_dispatch=float(np.clip((action_vec[1] + 1) / 2, 0, 1)),
            demand_shedding=float(np.clip((action_vec[2] + 1) / 2, 0, 1)),
        )

ppo_agent = PPOAgent(net)

def evaluate(agent_fn, name, tasks=('task_1_normal', 'task_2_heatwave', 'task_3_crisis'), seeds=(42, 7, 123)):
    rows = []
    for task in tasks:
        scores, blackouts = [], []
        for seed in seeds:
            env_eval = GridOpsEnvironment()
            traj, grade = run_episode(env_eval, agent_fn, task_id=task, seed=seed)
            scores.append(grade.get('score', 0) if grade else 0)
            blackouts.append(sum(t['blackout'] for t in traj))
        rows.append({'agent': name, 'task': task,
                     'score_mean': np.mean(scores), 'score_std': np.std(scores),
                     'blackout_mean': np.mean(blackouts)})
    return rows

results = []
results += evaluate(random_agent, 'random')
results += evaluate(heuristic_agent, 'heuristic')
results += evaluate(ppo_agent, 'ppo')
# results += evaluate(llm_agent, 'llm')   # slow/expensive — uncomment if API is configured

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
# Comparison bar chart — the demo visual
pivot = df.pivot(index='task', columns='agent', values='score_mean')
ax = pivot.plot(kind='bar', figsize=(10, 4))
ax.set_ylabel('Grader score (0–1)')
ax.set_title('Agent performance across GridOps tasks')
ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0); plt.legend(title='Agent')
plt.tight_layout(); plt.show()

---
## Part 9b — Overfitting checks

Before believing our numbers, three things to rule out:

| Failure mode | What it means | How we check |
|---|---|---|
| **Seed overfitting** | Policy memorised specific random seeds it saw during training and won't generalise to new ones | Evaluate on 30+ seeds drawn from a disjoint RNG. Report mean, std, min, and count of zero-scoring episodes. |
| **Train/eval divergence** | Training reward keeps going up while real eval score plateaus or drops | Already built in: the red eval curve in the training plot tracks alongside the blue training reward. They moved together → no divergence. |
| **Reward hacking** | Policy found a grader loophole that scores well but isn't realistic grid operation | Trajectory sanity check: plot the agent's SOC, diesel use, and blackouts vs the heuristic on the same seed. Structurally similar shapes = real operational logic. |

The two cells below run the first and third checks.

In [ ]:
def evaluate_robust(agent_fn, name, tasks=('task_1_normal', 'task_2_heatwave', 'task_3_crisis'), n_seeds=30):
    """Evaluate on many held-out seeds drawn from a deterministic generator.

    Training samples seeds from np.random.randint(1_000_000) — effectively
    any int. Here we use RandomState(999) to pick 30 integers that are
    reproducible across runs but (with overwhelming probability) disjoint
    from anything seen in training.

    We additionally track:
      - score_std:  how consistent the agent is across seeds (low = robust)
      - score_min:  worst single episode (a zero here is a catastrophic failure)
      - n_zero:     number of episodes where score < 0.05 (grader reliability gate triggered)
    """
    rng = np.random.RandomState(999)
    seeds = rng.randint(10_000, 1_000_000, size=n_seeds).tolist()
    rows = []
    for task in tasks:
        scores, blackouts = [], []
        for seed in seeds:
            env_eval = GridOpsEnvironment()
            traj, grade = run_episode(env_eval, agent_fn, task_id=task, seed=seed)
            scores.append(grade.get('score', 0) if grade else 0)
            blackouts.append(sum(t['blackout'] for t in traj))
        s = np.array(scores)
        rows.append({
            'agent': name, 'task': task,
            'score_mean': s.mean(),
            'score_std':  s.std(),
            'score_min':  s.min(),
            'n_zero':     int((s < 0.05).sum()),
            'blackout_mean': float(np.mean(blackouts)),
        })
    return rows

print(f'Evaluating on 30 held-out seeds per task per agent (180 episodes total)...')
robust_results = []
robust_results += evaluate_robust(heuristic_agent, 'heuristic')
robust_results += evaluate_robust(ppo_agent, 'ppo')

df_robust = pd.DataFrame(robust_results)
print('\nRobust evaluation (30 seeds per task):')
print(df_robust.to_string(index=False))
print('\nInterpretation:')
print('- Low score_std  → policy is consistent across varied conditions')
print('- High score_min → no catastrophic single-episode failures')
print('- n_zero == 0    → grader reliability gate never collapses the score')

In [ ]:
def compare_trajectories(task_id='task_3_crisis', seed=42):
    """Plot heuristic vs PPO on the same task+seed. A reality check: does
    the PPO policy do structurally sensible things (charge at night, discharge
    at peak, minimise diesel) or has it learned a grader-gaming shortcut?"""
    env_h = GridOpsEnvironment()
    traj_h, grade_h = run_episode(env_h, heuristic_agent, task_id=task_id, seed=seed)

    env_p = GridOpsEnvironment()
    traj_p, grade_p = run_episode(env_p, ppo_agent, task_id=task_id, seed=seed)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
    for ax, traj, label, grade in zip(
        axes, [traj_h, traj_p], ['Heuristic', 'PPO'], [grade_h, grade_p]
    ):
        hours = [t['hour'] for t in traj]
        ax.plot(hours, [t['demand'] for t in traj], 'r-', alpha=0.55, label='Demand (kW)')
        ax.plot(hours, [t['solar'] for t in traj],  color='gold', alpha=0.8, label='Solar (kW)')
        ax.fill_between(hours, 0, [t['blackout'] for t in traj],
                        color='red', alpha=0.35, label='Blackout (kW)')
        ax2 = ax.twinx()
        ax2.plot(hours, [t['soc']*100 for t in traj], 'b-', linewidth=2, label='SOC %')
        ax2.set_ylim(0, 100); ax2.set_ylabel('SOC (%)', color='blue')
        score = grade.get('score', 0.0) if grade else 0.0
        blackout_kwh = sum(t['blackout'] for t in traj)
        ax.set_ylabel('kW')
        ax.set_title(f'{label} on {task_id} (seed={seed})  —  score: {score:.3f}, blackout: {blackout_kwh:.1f} kWh')
        ax.legend(loc='upper left', fontsize=8)
        ax2.legend(loc='upper right', fontsize=8)
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel('Hour (step index)')
    plt.tight_layout(); plt.show()

# Compare on each of the three tasks — look for structural similarity in SOC patterns
for task in ['task_1_normal', 'task_2_heatwave', 'task_3_crisis']:
    compare_trajectories(task, seed=42)

---
## Part 10 — Hackathon playbook (the 48-hour plan)

You now have the full toolkit. Here is how to deploy it when Meta announces the Round 2 theme.

### Hour 0–4: scope ruthlessly
- Read the problem statement 3 times. Write down **what the grader rewards**.
- If an env is provided → skip to agent. If you must build one → reuse GridOps skeleton (physics.py structure, scenarios.py, graders.py).
- Pick **one** killer feature. Cut everything else.

### Hour 4–12: ship a random → heuristic → LLM pipeline
- Random first (proves plumbing).
- Heuristic next (proves you understand the domain).
- LLM third (instant non-trivial agent).
- **By hour 12, you should have a working demo.** Everything after this is improvement.

### Hour 12–30: RL training
- Wrap env in gym-style adapter (Part 7.1).
- Copy this notebook's PPO (Part 7.2–7.4).
- Train on simplest task first. Confirm learning curve goes up.
- If stuck after 2h: stop. Use heuristic + LLM for the demo. **Do not rabbit-hole.**

### Hour 30–42: polish
- **Learning curve plot** — non-negotiable. Proves training worked.
- **Comparison bar chart** (Part 9) — visceral evidence of improvement.
- **Live dashboard** — judges love seeing the agent act in real time.
- **README with ablation table** — random vs heuristic vs LLM vs RL.

### Hour 42–48: pitch
- 3-minute demo rehearsed **cold** (no slides, just run).
- Story arc: problem → env → baselines → training → result.
- End on the comparison chart.

---

### What judges weight (from Round 1 rubric, projected)

| Criterion | Weight | Your move |
|---|---|---|
| Real-world relevance | ~30% | GridOps already wins this; frame any new env the same way |
| Env + grader quality | ~25% | Reliability-gated grader, 3 escalating tasks |
| Agent quality | ~20% | RL > LLM > heuristic on the grader |
| Code quality | ~15% | Typed models, Docker, deterministic seeds |
| Demo & storytelling | ~10% | Learning curve + comparison chart |

---

### One last thing

At the finale, **the team that has a trained RL agent running live wins attention**. Most teams will ship LLM-only because it is easier. Your differentiator is showing the learning curve and the comparison chart.

This notebook is the scaffold. Run it end-to-end tonight. Tune the PPO hyperparameters. Train on Task 2. Ship it.